### Adaptive Question Difficulty with LLM
#
#### Objective
##### Dynamically generate quiz questions from **lecture content** at difficulty levels adapted to each student's wellbeing and academic performance. High-risk students receive easier questions to reduce stress, while low-risk students are challenged to maintain engagement.
#
#### How It Works
1.   Teacher uploads a lecture PDF
2.   System extracts and processes the content
3.   Student's risk label + exam score determines difficulty level
4.   Gemini Flash generates questions FROM the lecture at that difficulty
5.   System adapts difficulty based on student's quiz performance






#### Section 1: Install and Import Librares

In [ ]:
# ============================================================
# SECTION 1: INSTALL AND IMPORT LIBRARIES
# ============================================================
!pip install google-generativeai PyPDF2 pdfplumber pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
import time
import warnings
warnings.filterwarnings('ignore')

try:
    import pdfplumber
    HAS_PDF = True
    print("[OK] pdfplumber loaded")
except:
    HAS_PDF = False
    print("[WARN] pdfplumber not found. Install: pip install pdfplumber")

try:
    import google.generativeai as genai
    HAS_GENAI = True
    print("[OK] google-generativeai loaded")
except:
    HAS_GENAI = False
    print("[WARN] google-generativeai not found. Install: pip install google-generativeai")

plt.style.use('seaborn-v0_8-whitegrid')

print("\n" + "=" * 60)
print("COLAB 2: ADAPTIVE QUESTION DIFFICULTY WITH LLM")
print("=" * 60)

 #### Section 2: Configure GEMINI API

In [ ]:
# ============================================================
# SECTION 2: CONFIGURE GEMINI API
# ============================================================
# Replace with your actual API key
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"

if HAS_GENAI and GEMINI_API_KEY != "YOUR_KEY_HERE":
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel('gemini-2.5-flash')
    print("[OK] Gemini Flash API configured successfully")
else:
    model = None
    print("[INFO] Gemini API not configured. Set GEMINI_API_KEY above.")
    print("       Demo mode will use sample questions instead.")


#### Section 3: PDF Lecture Content Extraction




In [ ]:
# ============================================================
# SECTION 3: PDF LECTURE CONTENT EXTRACTION
# ============================================================
from google.colab import files; uploaded = files.upload()
# Then set LECTURE_PDF_PATH to the uploaded file name

LECTURE_PDF_PATH = "lecture_slides.pdf"  # Update with your PDF filename

def extract_pdf_content(pdf_path):
    """Extract text content from a PDF file using pdfplumber"""
    if not HAS_PDF:
        print("[WARN] pdfplumber not available. Using sample content.")
        return None, None

    try:
        pages_content = []
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text()
                if text:
                    # Clean the text
                    text = re.sub(r'\s+', ' ', text).strip()
                    text = re.sub(r'Page \d+', '', text)  # Remove page numbers
                    pages_content.append({
                        'page': i + 1,
                        'content': text,
                        'word_count': len(text.split())
                    })

        full_content = ' '.join([p['content'] for p in pages_content])
        print(f"[OK] Extracted {len(pages_content)} pages")
        print(f"     Total words: {sum(p['word_count'] for p in pages_content)}")
        return pages_content, full_content
    except FileNotFoundError:
        print(f"[WARN] PDF file '{pdf_path}' not found. Using sample content.")
        return None, None

pages, full_content = extract_pdf_content(LECTURE_PDF_PATH)

# --- FALLBACK: Sample lecture content for demo ---
# This is used when no PDF is uploaded, so the notebook still runs end-to-end

SAMPLE_LECTURE_CONTENT = """
Introduction to Photosynthesis - Grade 8 Science

Photosynthesis is the process by which green plants and some other organisms use
sunlight to synthesize foods with carbon dioxide and water. Photosynthesis in plants
generally involves the green pigment chlorophyll and generates oxygen as a byproduct.

The Chemical Equation:
6CO2 + 6H2O + light energy -> C6H12O6 + 6O2

Key Concepts:
1. Light-dependent reactions occur in the thylakoid membranes
2. Light-independent reactions (Calvin Cycle) occur in the stroma
3. Chlorophyll absorbs red and blue light, reflects green light
4. Factors affecting photosynthesis: light intensity, CO2 concentration, temperature
5. Plants use glucose for energy, growth, and storage as starch

Importance of Photosynthesis:
- Produces oxygen for all living organisms
- Forms the base of most food chains
- Removes carbon dioxide from the atmosphere
- Provides raw materials for plant growth

Experiment: Testing for Starch in Leaves
Materials: leaf, iodine solution, ethanol, boiling water
Method: Boil leaf in water, then in ethanol to remove chlorophyll.
Add iodine solution - blue-black color indicates presence of starch.

Summary: Photosynthesis is essential for life on Earth. It converts light energy
into chemical energy stored in glucose, releasing oxygen as a byproduct.
"""

if full_content is None:
    full_content = SAMPLE_LECTURE_CONTENT
    pages = [{'page': 1, 'content': SAMPLE_LECTURE_CONTENT, 'word_count': len(SAMPLE_LECTURE_CONTENT.split())}]
    print("\n[DEMO] Using sample lecture content: 'Introduction to Photosynthesis'")

# --- Visualize PDF content statistics ---
pages_df = pd.DataFrame(pages)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(pages_df['page'], pages_df['word_count'], color='#3498db', edgecolor='white')
axes[0].set_title('Word Count per Slide/Page', fontweight='bold')
axes[0].set_xlabel('Page Number')
axes[0].set_ylabel('Word Count')

# Topic extraction (keyword frequency)
words = full_content.lower().split()
stop_words = {'the', 'is', 'in', 'and', 'of', 'to', 'a', 'for', 'that', 'are', 'by', 'as', 'it', 'from', 'with', 'on', 'an'}
keywords = [w for w in words if len(w) > 3 and w not in stop_words and w.isalpha()]
from collections import Counter
top_keywords = Counter(keywords).most_common(15)

kw_names, kw_counts = zip(*top_keywords) if top_keywords else ([], [])
axes[1].barh(kw_names, kw_counts, color='#9b59b6', edgecolor='white')
axes[1].set_title('Top Keywords in Lecture Content', fontweight='bold')
axes[1].set_xlabel('Frequency')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('c2_chart1_pdf_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


#### Section 4: Student Difficulty Mapping

In [ ]:
# ============================================================
# SECTION 4: LOAD STUDENT DATA & MAP DIFFICULTY LEVELS
# ============================================================
df = pd.read_csv('student_wellbeing_dataset.csv')

# Use only the LATEST week (week 8) data for each student
# This represents their current state
latest = df[df['week'] == df.groupby('student_id')['week'].transform('max')]
latest = latest.copy()

print(f"Students: {latest.shape[0]}")
print(f"\nRisk distribution:")
print(latest['risk_label'].value_counts())

# %%
# --- Performance Level Binning ---
# Unique to this notebook: bin continuous scores into categorical levels
def assign_performance_level(score):
    if score < 50:
        return 'Low'
    elif score <= 75:
        return 'Mid'
    else:
        return 'High'

latest['performance_level'] = latest['avg_exam_score'].apply(assign_performance_level)

# --- Difficulty Mapping ---
difficulty_map = {
    ('HIGH', 'Low'): 'VERY_EASY',
    ('HIGH', 'Mid'): 'EASY',
    ('HIGH', 'High'): 'MEDIUM',
    ('MEDIUM', 'Low'): 'EASY',
    ('MEDIUM', 'Mid'): 'MEDIUM',
    ('MEDIUM', 'High'): 'MEDIUM_HARD',
    ('LOW', 'Low'): 'MEDIUM',
    ('LOW', 'Mid'): 'MEDIUM_HARD',
    ('LOW', 'High'): 'HARD',
}

# Bloom's taxonomy mapping
blooms_map = {
    'VERY_EASY': 'Remember',
    'EASY': 'Understand',
    'MEDIUM': 'Apply',
    'MEDIUM_HARD': 'Analyze',
    'HARD': 'Evaluate/Create'
}

latest['difficulty_level'] = latest.apply(
    lambda r: difficulty_map.get((r['risk_label'], r['performance_level']), 'MEDIUM'), axis=1
)
latest['bloom_level'] = latest['difficulty_level'].map(blooms_map)

print("\n--- Difficulty Assignment Summary ---")
print(latest['difficulty_level'].value_counts())
print(f"\nSample assignments:")
print(latest[['student_id', 'grade', 'risk_label', 'avg_exam_score', 'performance_level',
               'difficulty_level', 'bloom_level']].head(10).to_string(index=False))

# %%
# --- Visualization: Difficulty Distribution ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Difficulty distribution
diff_counts = latest['difficulty_level'].value_counts()
diff_order = ['VERY_EASY', 'EASY', 'MEDIUM', 'MEDIUM_HARD', 'HARD']
diff_colors = ['#2ecc71', '#27ae60', '#f39c12', '#e67e22', '#e74c3c']
counts_ordered = [diff_counts.get(d, 0) for d in diff_order]

axes[0].bar(diff_order, counts_ordered, color=diff_colors, edgecolor='white')
axes[0].set_title('Difficulty Level Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Students')
axes[0].tick_params(axis='x', rotation=45)

# Chart 2: Risk x Performance Heatmap
ct = pd.crosstab(latest['risk_label'], latest['performance_level'])
ct = ct.reindex(index=['HIGH', 'MEDIUM', 'LOW'], columns=['Low', 'Mid', 'High'])
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Risk Level x Performance Level', fontweight='bold')

# Chart 3: Students per grade
grade_counts = latest.groupby(['grade', 'difficulty_level']).size().unstack(fill_value=0)
grade_counts.plot(kind='bar', stacked=True, ax=axes[2], color=diff_colors[:len(grade_counts.columns)])
axes[2].set_title('Difficulty by Grade', fontweight='bold')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('c2_chart2_difficulty_mapping.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 60)
print("PART 1 COMPLETE: PDF extracted & difficulty mapped!")
print("Proceed to Part 2 for Quiz Generation & Analysis.")
print("=" * 60)

#### Section 5: Gemini Flash Quiz Generation Pipeline

In [ ]:
# ============================================================
# SETUP (Run after Part 1 or standalone)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
import time
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

try:
    import google.generativeai as genai
    HAS_GENAI = True
except: HAS_GENAI = False

plt.style.use('seaborn-v0_8-whitegrid')

# --- CONFIGURE API (set your key) ---
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"

if HAS_GENAI and GEMINI_API_KEY != "YOUR_KEY_HERE":
    genai.configure(api_key=GEMINI_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-2.5-flash')
    API_AVAILABLE = True
else:
    gemini_model = None
    API_AVAILABLE = False
    print("[INFO] Running in DEMO mode (no API key). Using sample questions.")

# --- Load data & recreate mappings ---
df = pd.read_csv('student_wellbeing_dataset.csv')
latest = df[df['week'] == df.groupby('student_id')['week'].transform('max')].copy()

def assign_perf(s):
    return 'Low' if s < 50 else ('Mid' if s <= 75 else 'High')

latest['performance_level'] = latest['avg_exam_score'].apply(assign_perf)

diff_map = {
    ('HIGH','Low'):'VERY_EASY', ('HIGH','Mid'):'EASY', ('HIGH','High'):'MEDIUM',
    ('MEDIUM','Low'):'EASY', ('MEDIUM','Mid'):'MEDIUM', ('MEDIUM','High'):'MEDIUM_HARD',
    ('LOW','Low'):'MEDIUM', ('LOW','Mid'):'MEDIUM_HARD', ('LOW','High'):'HARD',
}
blooms = {'VERY_EASY':'Remember','EASY':'Understand','MEDIUM':'Apply',
          'MEDIUM_HARD':'Analyze','HARD':'Evaluate/Create'}

latest['difficulty_level'] = latest.apply(lambda r: diff_map.get((r['risk_label'],r['performance_level']),'MEDIUM'), axis=1)
latest['bloom_level'] = latest['difficulty_level'].map(blooms)

# Sample lecture content
SAMPLE_LECTURE_CONTENT = """
Introduction to Photosynthesis - Grade 8 Science
Photosynthesis is the process by which green plants use sunlight to synthesize foods
with carbon dioxide and water. The chemical equation: 6CO2 + 6H2O + light -> C6H12O6 + 6O2.
Key concepts: Light-dependent reactions in thylakoid membranes, Calvin Cycle in stroma,
chlorophyll absorbs red and blue light. Factors: light intensity, CO2, temperature.
Plants use glucose for energy, growth, and starch storage. Photosynthesis produces oxygen,
forms the base of food chains, and removes CO2 from atmosphere.
Experiment: Test for starch using iodine solution - blue-black indicates starch.
"""

# Use extracted PDF content if available (from Part 1 / Section 3),
# otherwise fall back to the sample photosynthesis content
try:
    if full_content is not None and len(full_content.strip()) > 0:
        LECTURE_CONTENT = full_content
        print("[OK] Using extracted PDF lecture content for quiz generation.")
    else:
        LECTURE_CONTENT = SAMPLE_LECTURE_CONTENT
        print("[INFO] No PDF content found. Using sample lecture content (Photosynthesis).")
except NameError:
    LECTURE_CONTENT = SAMPLE_LECTURE_CONTENT
    print("[INFO] 'full_content' not defined. Using sample lecture content (Photosynthesis).")
    print("       Run Part 1 (Section 3: PDF Extraction) first to use your own lecture PDF.")


print("=" * 60)
print("SECTION 5: QUIZ GENERATION PIPELINE")
print("=" * 60)

In [ ]:
# ============================================================
# QUIZ GENERATION FUNCTION
# ============================================================
def generate_quiz_gemini(lecture_content, grade, difficulty, bloom_level, n_questions=5):
    """Generate quiz questions from lecture content using Gemini Flash API"""

    prompt = f"""You are an educational quiz generator for Sri Lankan school students.

LECTURE CONTENT:
{lecture_content}

STUDENT CONTEXT:
- Grade: {grade}
- Difficulty Level: {difficulty}
- Bloom's Taxonomy Target: {bloom_level}

TASK:
Generate exactly {n_questions} multiple-choice questions STRICTLY based on the lecture content above.
Each question must:
- Be derived from the lecture material (not general knowledge)
- Match the {bloom_level} level of Bloom's taxonomy
- Be appropriate for Grade {grade} students in Sri Lanka
- Include 4 options (A, B, C, D) with exactly one correct answer
- Include a brief explanation referencing the lecture content

OUTPUT FORMAT: Return ONLY a valid JSON array. No markdown, no extra text.
[
  {{
    "question": "...",
    "option_a": "...",
    "option_b": "...",
    "option_c": "...",
    "option_d": "...",
    "correct_answer": "A/B/C/D",
    "explanation": "...",
    "bloom_level": "{bloom_level}"
  }}
]"""

    if not API_AVAILABLE:
        return None

    try:
        response = gemini_model.generate_content(prompt)
        text = response.text.strip()
        # Clean markdown code blocks if present
        text = re.sub(r'```json\s*', '', text)
        text = re.sub(r'```\s*', '', text)
        questions = json.loads(text)
        return questions
    except Exception as e:
        print(f"  [ERROR] API call failed: {e}")
        return None

# %%
# --- SAMPLE QUESTIONS (Fallback for demo mode) ---
SAMPLE_QUESTIONS = {
    'VERY_EASY': [
        {"question": "What is photosynthesis?", "option_a": "Process of making food using sunlight", "option_b": "Process of breathing", "option_c": "Process of digestion", "option_d": "Process of excretion", "correct_answer": "A", "explanation": "As stated in the lecture, photosynthesis is the process by which green plants use sunlight to synthesize foods.", "bloom_level": "Remember"},
        {"question": "True or False: Chlorophyll is green in color.", "option_a": "True", "option_b": "False", "option_c": "Sometimes", "option_d": "Only in summer", "correct_answer": "A", "explanation": "The lecture states chlorophyll reflects green light, making it appear green.", "bloom_level": "Remember"},
    ],
    'EASY': [
        {"question": "Explain why leaves appear green.", "option_a": "Chlorophyll absorbs green light", "option_b": "Chlorophyll reflects green light", "option_c": "Leaves produce green pigment for protection", "option_d": "Green is the natural color of cells", "correct_answer": "B", "explanation": "The lecture explains that chlorophyll absorbs red and blue light and reflects green light.", "bloom_level": "Understand"},
        {"question": "What are the raw materials needed for photosynthesis?", "option_a": "Oxygen and glucose", "option_b": "Carbon dioxide and water", "option_c": "Nitrogen and hydrogen", "option_d": "Starch and minerals", "correct_answer": "B", "explanation": "The lecture's chemical equation shows CO2 and H2O as inputs.", "bloom_level": "Understand"},
    ],
    'MEDIUM': [
        {"question": "If a plant is placed in a dark room for 48 hours, what happens to starch production?", "option_a": "Starch increases", "option_b": "Starch stays the same", "option_c": "Starch production stops", "option_d": "Starch converts to protein", "correct_answer": "C", "explanation": "Light is required for photosynthesis. Without light, the plant cannot produce glucose or starch.", "bloom_level": "Apply"},
        {"question": "Using the equation from the lecture, how many molecules of water are needed to produce one glucose molecule?", "option_a": "2", "option_b": "4", "option_c": "6", "option_d": "12", "correct_answer": "C", "explanation": "The equation 6CO2 + 6H2O -> C6H12O6 + 6O2 shows 6 water molecules are needed.", "bloom_level": "Apply"},
    ],
    'MEDIUM_HARD': [
        {"question": "Compare and contrast the light-dependent and light-independent reactions.", "option_a": "Both occur in the stroma", "option_b": "Light-dependent in thylakoids, light-independent in stroma", "option_c": "Both require direct sunlight", "option_d": "Light-independent occurs first", "correct_answer": "B", "explanation": "The lecture states light-dependent reactions occur in thylakoid membranes and Calvin Cycle in stroma.", "bloom_level": "Analyze"},
    ],
    'HARD': [
        {"question": "Evaluate how increasing temperature beyond 40C would affect photosynthesis rate and explain why.", "option_a": "Rate increases because reactions speed up", "option_b": "Rate decreases because enzymes denature", "option_c": "No effect on the rate", "option_d": "Rate doubles with each 10C increase", "correct_answer": "B", "explanation": "Temperature is a factor affecting photosynthesis. Enzymes involved denature at high temperatures, reducing the rate.", "bloom_level": "Evaluate/Create"},
    ],
}

In [ ]:
# ============================================================
# GENERATE QUESTIONS FOR SAMPLE STUDENTS
# ============================================================
print("\n--- Generating Questions for Sample Students ---")

# Select 15 students: 5 per risk level
sample_students = []
for risk in ['LOW', 'MEDIUM', 'HIGH']:
    risk_students = latest[latest['risk_label'] == risk].head(2) #HEAD * 3 = STUDENTS
    sample_students.append(risk_students)
sample_df = pd.concat(sample_students)

all_questions = []

for _, student in sample_df.iterrows():
    sid = student['student_id']
    grade = student['grade']
    difficulty = student['difficulty_level']
    bloom = student['bloom_level']

    print(f"\n  Student {sid} | Grade {grade} | Risk: {student['risk_label']} | "
          f"Difficulty: {difficulty} | Bloom: {bloom}")

    if API_AVAILABLE:
        questions = generate_quiz_gemini(LECTURE_CONTENT, grade, difficulty, bloom, n_questions=3) #STUDENTS * QUESTIONS =TOTAl_QUESTIONS
        time.sleep(1)  # Rate limiting
    else:
        questions = SAMPLE_QUESTIONS.get(difficulty, SAMPLE_QUESTIONS['MEDIUM'])[:3]

    if questions:
        for q in questions:
            q['student_id'] = sid
            q['grade'] = grade
            q['risk_label'] = student['risk_label']
            q['difficulty_level'] = difficulty
            all_questions.append(q)
        print(f"    -> Generated {len(questions)} questions")
    else:
        print(f"    -> [SKIP] No questions generated")

questions_df = pd.DataFrame(all_questions)
print(f"\n[OK] Total questions generated: {len(questions_df)}")
questions_df.head()


#### Section 6: Quality Analysis of Generated Questions

In [ ]:
# ============================================================
# QUALITY ANALYSIS
# ============================================================
print("\n" + "=" * 60)
print("SECTION 6: QUALITY ANALYSIS")
print("=" * 60)

# --- Content Fidelity Check ---
# Check if question keywords appear in the lecture content
lecture_words = set(LECTURE_CONTENT.lower().split())

def content_fidelity(question_text):
    q_words = set(question_text.lower().split())
    overlap = q_words.intersection(lecture_words)
    return len(overlap) / max(len(q_words), 1)

if len(questions_df) > 0:
    questions_df['content_fidelity'] = questions_df['question'].apply(content_fidelity)
    questions_df['word_count'] = questions_df['question'].apply(lambda x: len(str(x).split()))

    print(f"\nContent Fidelity Score: {questions_df['content_fidelity'].mean():.3f} (1.0 = perfect overlap)")
    print(f"Avg Question Length: {questions_df['word_count'].mean():.1f} words")

# %%
# --- Quality Charts ---
if len(questions_df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Chart 1: Bloom's level distribution
    bloom_counts = questions_df['bloom_level'].value_counts()
    axes[0, 0].pie(bloom_counts, labels=bloom_counts.index, autopct='%1.1f%%',
                   startangle=90, colors=plt.cm.Set3(np.linspace(0, 1, len(bloom_counts))))
    axes[0, 0].set_title("Bloom's Level Distribution", fontweight='bold')

    # Chart 2: Word count by difficulty
    diff_order = ['VERY_EASY', 'EASY', 'MEDIUM', 'MEDIUM_HARD', 'HARD']
    existing_diffs = [d for d in diff_order if d in questions_df['difficulty_level'].unique()]
    data_wc = [questions_df[questions_df['difficulty_level'] == d]['word_count'].values for d in existing_diffs]
    if data_wc:
        bp = axes[0, 1].boxplot(data_wc, labels=existing_diffs, patch_artist=True)
        colors_bp = ['#2ecc71', '#27ae60', '#f39c12', '#e67e22', '#e74c3c']
        for patch, c in zip(bp['boxes'], colors_bp[:len(bp['boxes'])]):
            patch.set_facecolor(c); patch.set_alpha(0.7)
    axes[0, 1].set_title('Word Count by Difficulty', fontweight='bold')
    axes[0, 1].tick_params(axis='x', rotation=45)

    # Chart 3: Content fidelity by difficulty
    fidelity_by_diff = questions_df.groupby('difficulty_level')['content_fidelity'].mean()
    fidelity_by_diff = fidelity_by_diff.reindex([d for d in diff_order if d in fidelity_by_diff.index])
    axes[0, 2].bar(fidelity_by_diff.index, fidelity_by_diff.values,
                   color='#3498db', edgecolor='white')
    axes[0, 2].set_title('Content Fidelity by Difficulty', fontweight='bold')
    axes[0, 2].set_ylabel('Fidelity Score')
    axes[0, 2].tick_params(axis='x', rotation=45)

    # Chart 4: Questions per risk category
    risk_q = questions_df.groupby(['risk_label', 'difficulty_level']).size().unstack(fill_value=0)
    risk_q.plot(kind='bar', ax=axes[1, 0], edgecolor='white')
    axes[1, 0].set_title('Questions per Risk Category', fontweight='bold')
    axes[1, 0].tick_params(axis='x', rotation=0)
    axes[1, 0].legend(fontsize=7)

    # Chart 5: Questions per grade
    grade_q = questions_df['grade'].value_counts().sort_index()
    axes[1, 1].bar(grade_q.index, grade_q.values, color='#9b59b6', edgecolor='white')
    axes[1, 1].set_title('Questions per Grade Level', fontweight='bold')
    axes[1, 1].set_xlabel('Grade')

    # Chart 6: Difficulty distribution
    diff_q = questions_df['difficulty_level'].value_counts()
    diff_q = diff_q.reindex([d for d in diff_order if d in diff_q.index])
    axes[1, 2].bar(diff_q.index, diff_q.values,
                   color=['#2ecc71','#27ae60','#f39c12','#e67e22','#e74c3c'][:len(diff_q)],
                   edgecolor='white')
    axes[1, 2].set_title('Question Difficulty Distribution', fontweight='bold')
    axes[1, 2].tick_params(axis='x', rotation=45)

    fig.suptitle('Quiz Generation Quality Analysis', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('c2_chart3_quality_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

#### Section 7: Adaptive Difficulty Simulation

In [ ]:
# ============================================================
# SECTION 7: SIMULATE ADAPTIVE RESPONSE LOOP (3 ROUNDS)
# ============================================================
print("\n" + "=" * 60)
print("SECTION 7: ADAPTIVE DIFFICULTY SIMULATION")
print("=" * 60)

difficulty_levels = ['VERY_EASY', 'EASY', 'MEDIUM', 'MEDIUM_HARD', 'HARD']

# Probability of correct answer based on risk and difficulty
# Higher risk students perform better on easier questions
correct_probs = {
    'LOW':    {'VERY_EASY': 0.95, 'EASY': 0.90, 'MEDIUM': 0.80, 'MEDIUM_HARD': 0.70, 'HARD': 0.60},
    'MEDIUM': {'VERY_EASY': 0.90, 'EASY': 0.80, 'MEDIUM': 0.65, 'MEDIUM_HARD': 0.50, 'HARD': 0.35},
    'HIGH':   {'VERY_EASY': 0.85, 'EASY': 0.70, 'MEDIUM': 0.50, 'MEDIUM_HARD': 0.30, 'HARD': 0.15},
}

# Simulate 3 rounds for each sample student
np.random.seed(42)
simulation_results = []

for _, student in sample_df.iterrows():
    sid = student['student_id']
    risk = student['risk_label']
    current_diff = student['difficulty_level'] if 'difficulty_level' in student.index else 'MEDIUM'
    diff_idx = difficulty_levels.index(current_diff) if current_diff in difficulty_levels else 2

    for round_num in range(1, 4):
        current_diff = difficulty_levels[diff_idx]
        prob = correct_probs[risk][current_diff]

        # Simulate 5 question answers
        n_correct = np.random.binomial(5, prob)
        accuracy = n_correct / 5

        simulation_results.append({
            'student_id': sid, 'risk_label': risk, 'round': round_num,
            'difficulty': current_diff, 'accuracy': accuracy, 'n_correct': n_correct
        })

        # Adapt difficulty for next round
        if accuracy < 0.5:
            diff_idx = max(0, diff_idx - 1)  # Easier
        elif accuracy > 0.8:
            diff_idx = min(4, diff_idx + 1)  # Harder

sim_df = pd.DataFrame(simulation_results)

print(f"Simulation: {len(sample_df)} students x 3 rounds = {len(sim_df)} records")
print(f"\nAverage accuracy by round:")
print(sim_df.groupby('round')['accuracy'].mean().round(3))

# %%
# --- Simulation Charts ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Chart 1: Accuracy by difficulty level
acc_by_diff = sim_df.groupby('difficulty')['accuracy'].mean()
acc_by_diff = acc_by_diff.reindex([d for d in difficulty_levels if d in acc_by_diff.index])
axes[0, 0].bar(acc_by_diff.index, acc_by_diff.values,
               color=['#2ecc71','#27ae60','#f39c12','#e67e22','#e74c3c'][:len(acc_by_diff)],
               edgecolor='white')
axes[0, 0].set_title('Average Accuracy by Difficulty Level', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')
axes[0, 0].legend()

# Chart 2: Accuracy improvement across rounds
for risk in ['LOW', 'MEDIUM', 'HIGH']:
    risk_data = sim_df[sim_df['risk_label'] == risk]
    acc_by_round = risk_data.groupby('round')['accuracy'].mean()
    color = {'LOW': '#2ecc71', 'MEDIUM': '#f39c12', 'HIGH': '#e74c3c'}[risk]
    axes[0, 1].plot(acc_by_round.index, acc_by_round.values, 'o-',
                    linewidth=2, markersize=8, color=color, label=f'{risk} Risk')
axes[0, 1].set_title('Accuracy Improvement Across Rounds', fontweight='bold')
axes[0, 1].set_xlabel('Round')
axes[0, 1].set_ylabel('Average Accuracy')
axes[0, 1].set_xticks([1, 2, 3])
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Chart 3: Difficulty trajectory (5 sample students)
sample_sids = sim_df['student_id'].unique()[:5]
colors_line = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']
for i, sid in enumerate(sample_sids):
    s_data = sim_df[sim_df['student_id'] == sid]
    diff_nums = [difficulty_levels.index(d) for d in s_data['difficulty']]
    axes[1, 0].step(s_data['round'], diff_nums, where='mid', linewidth=2,
                    marker='o', color=colors_line[i], label=f'{sid} ({s_data["risk_label"].iloc[0]})')
axes[1, 0].set_yticks(range(5))
axes[1, 0].set_yticklabels(difficulty_levels, fontsize=8)
axes[1, 0].set_title('Adaptive Difficulty Trajectory (5 Students)', fontweight='bold')
axes[1, 0].set_xlabel('Round')
axes[1, 0].set_ylabel('Difficulty Level')
axes[1, 0].legend(fontsize=7)

# Chart 4: Stress reduction estimate
# Compare average accuracy with adapted vs fixed difficulty
fixed_acc = sim_df[sim_df['round'] == 1]['accuracy'].mean()
adapted_acc = sim_df[sim_df['round'] == 3]['accuracy'].mean()
axes[1, 1].bar(['Fixed Difficulty\n(Round 1)', 'Adapted Difficulty\n(Round 3)'],
               [fixed_acc, adapted_acc], color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[1, 1].set_title('Accuracy: Fixed vs Adapted Difficulty', fontweight='bold')
axes[1, 1].set_ylabel('Average Accuracy')
improvement = (adapted_acc - fixed_acc) * 100
axes[1, 1].text(1, adapted_acc + 0.02, f'+{improvement:.1f}%', ha='center', fontweight='bold', fontsize=12)

fig.suptitle('Adaptive Difficulty Simulation Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('c2_chart4_adaptive_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

#### Section 8: End-to-End Pipeline Demo & Export

In [ ]:
# ============================================================
# END-TO-END PIPELINE DEMO
# ============================================================
print("\n" + "=" * 60)
print("END-TO-END PIPELINE DEMO")
print("=" * 60)

demo_student = latest.iloc[0]
print(f"\n1. Student: {demo_student['student_id']}")
print(f"2. Grade: {demo_student['grade']}")
print(f"3. Wellbeing Score: {demo_student['composite_wellbeing_score']}")
print(f"4. Risk Label: {demo_student['risk_label']}")
print(f"5. Exam Score: {demo_student['avg_exam_score']} -> Performance: {demo_student['performance_level']}")
print(f"6. Assigned Difficulty: {demo_student['difficulty_level']}")
print(f"7. Bloom's Target: {demo_student['bloom_level']}")
print(f"\n8. Generated Quiz:")

demo_qs = questions_df[questions_df['student_id'] == demo_student['student_id']]
if len(demo_qs) > 0:
    for i, (_, q) in enumerate(demo_qs.iterrows(), 1):
        print(f"\n   Q{i}. {q['question']}")
        print(f"       A) {q['option_a']}")
        print(f"       B) {q['option_b']}")
        print(f"       C) {q['option_c']}")
        print(f"       D) {q['option_d']}")
        print(f"       Correct: {q['correct_answer']}")
        print(f"       Explanation: {q['explanation']}")
else:
    print("   [No questions available for this student]")

# %%
# --- Export ---
if len(questions_df) > 0:
    questions_df.to_csv('generated_quiz_questions.csv', index=False)
    print(f"\n[OK] Questions exported: generated_quiz_questions.csv ({len(questions_df)} questions)")

sim_df.to_csv('adaptive_simulation_results.csv', index=False)
print(f"[OK] Simulation exported: adaptive_simulation_results.csv ({len(sim_df)} records)")

print("\n" + "=" * 60)
print("COLAB 2 COMPLETE: Adaptive Question Difficulty System built!")
print("=" * 60)
print("\nAll charts saved:")
print("  - c2_chart1_pdf_analysis.png")
print("  - c2_chart2_difficulty_mapping.png")
print("  - c2_chart3_quality_analysis.png")
print("  - c2_chart4_adaptive_simulation.png")
